# Neo4j Graph Database Integration

This notebook demonstrates how to work with topologic_fast graphs and Neo4j graph database.

**Note:** Native Neo4j integration is not yet implemented in topologic_fast. This notebook shows:
1. How to create topological graphs using topologic_fast
2. How to extract graph data for export to Neo4j
3. Example Cypher queries that would be used with the exported data

## What is Neo4j?

Neo4j is a native graph database that stores and manages data as nodes and relationships. It's ideal for:
- Building information modeling (BIM)
- Spatial relationship queries
- Network analysis
- Knowledge graphs

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import json

# Check if neo4j driver is available
try:
    from neo4j import GraphDatabase
    NEO4J_AVAILABLE = True
    print("neo4j driver is available")
except ImportError:
    NEO4J_AVAILABLE = False
    print("neo4j driver not installed. Install with: pip install neo4j")
    print("This notebook will still demonstrate the graph structure for export.")

## 1. Create a Topologic CellComplex

Let's create a building-like structure with multiple cells (rooms).

In [ ]:
# Create a simple building with multiple rooms
# Ground floor rooms
room_a = tf.Cell.Box(0, 0, 0, 5, 5, 3)  # Room A
room_b = tf.Cell.Box(5, 0, 0, 5, 5, 3)  # Room B (adjacent to A)
room_c = tf.Cell.Box(0, 5, 0, 5, 5, 3)  # Room C (adjacent to A)
room_d = tf.Cell.Box(5, 5, 0, 5, 5, 3)  # Room D (adjacent to B, C)

# Upper floor rooms (stacked on ground floor)
room_e = tf.Cell.Box(0, 0, 3, 10, 5, 3)  # Room E (above A, B)
room_f = tf.Cell.Box(0, 5, 3, 10, 5, 3)  # Room F (above C, D)

# Combine into a CellComplex
building = tf.CellComplex.ByCells([room_a, room_b, room_c, room_d, room_e, room_f])

print(f"Building Statistics:")
print(f"  Cells (rooms): {building.NumCells()}")
print(f"  Total Volume: {building.Volume():.1f} m^3")

## 2. Create a Dual Graph

The dual graph represents room connectivity:
- **Vertices** = Room centroids  
- **Edges** = Shared walls/floors between rooms

In [ ]:
# Create the dual graph
graph = tf.Graph.ByTopology(building)

print(f"Connectivity Graph:")
print(f"  Vertices (rooms): {graph.Order()}")
print(f"  Edges (connections): {graph.Size()}")
print(f"  Density: {graph.Density():.3f}")
print(f"  Diameter: {graph.Diameter()} steps")

## 3. Assign Room Metadata

In topologicpy, we would use Dictionary to attach metadata. For topologic_fast, we'll track this separately.

In [ ]:
# Room metadata (in topologicpy this would be attached via Dictionary)
# NOTE: tf.Dictionary is not yet implemented in topologic_fast

room_metadata = [
    {"label": "Room_A", "type": "office", "floor": 0, "area": 25.0},
    {"label": "Room_B", "type": "office", "floor": 0, "area": 25.0},
    {"label": "Room_C", "type": "conference", "floor": 0, "area": 25.0},
    {"label": "Room_D", "type": "kitchen", "floor": 0, "area": 25.0},
    {"label": "Room_E", "type": "open_space", "floor": 1, "area": 50.0},
    {"label": "Room_F", "type": "open_space", "floor": 1, "area": 50.0},
]

# Get graph vertices and their coordinates
graph_vertices = graph.Vertices()
print("Room Vertices:")
for i, v in enumerate(graph_vertices):
    coords = v.Coordinates()
    meta = room_metadata[i]
    print(f"  {meta['label']}: position=({coords[0]:.1f}, {coords[1]:.1f}, {coords[2]:.1f}), type={meta['type']}")

## 4. Extract Graph Structure for Neo4j

Let's extract the nodes and relationships in a format suitable for Neo4j.

In [ ]:
def get_vertex_index(graph, vertex, vertices):
    """Find the index of a vertex in the graph."""
    target_coords = vertex.Coordinates()
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        if (abs(coords[0] - target_coords[0]) < 0.01 and
            abs(coords[1] - target_coords[1]) < 0.01 and
            abs(coords[2] - target_coords[2]) < 0.01):
            return i
    return -1

# Extract nodes
nodes = []
for i, v in enumerate(graph_vertices):
    coords = v.Coordinates()
    node = {
        "id": i,
        "x": coords[0],
        "y": coords[1],
        "z": coords[2],
        **room_metadata[i]
    }
    nodes.append(node)

# Extract edges (relationships)
edges = graph.Edges()
relationships = []

for edge in edges:
    edge_verts = edge.Vertices()
    if len(edge_verts) == 2:
        start_idx = get_vertex_index(graph, edge_verts[0], graph_vertices)
        end_idx = get_vertex_index(graph, edge_verts[1], graph_vertices)
        
        # Determine relationship type based on Z coordinates
        start_z = edge_verts[0].Coordinates()[2]
        end_z = edge_verts[1].Coordinates()[2]
        
        if abs(start_z - end_z) > 0.1:
            rel_type = "STACKED_ON"  # Vertical relationship
        else:
            rel_type = "ADJACENT_TO"  # Horizontal relationship
        
        relationships.append({
            "start": start_idx,
            "end": end_idx,
            "type": rel_type,
            "start_label": room_metadata[start_idx]["label"],
            "end_label": room_metadata[end_idx]["label"]
        })

print("Nodes for Neo4j:")
for node in nodes:
    print(f"  {node}")

print("\nRelationships for Neo4j:")
for rel in relationships:
    print(f"  ({rel['start_label']})-[:{rel['type']}]->({rel['end_label']})")

## 5. Visualize the Graph

In [ ]:
def visualize_graph_3d(graph, metadata):
    """Visualize the graph in 3D."""
    fig = go.Figure()
    
    # Color by room type
    type_colors = {
        'office': 'green',
        'conference': 'gold',
        'kitchen': 'orange',
        'open_space': 'lightblue'
    }
    
    # Draw edges
    edges = graph.Edges()
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            
            # Color by relationship type
            if abs(p1[2] - p2[2]) > 0.1:
                color = 'red'  # Vertical
                width = 4
            else:
                color = 'blue'  # Horizontal
                width = 3
            
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color=color, width=width),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices
    vertices = graph.Vertices()
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        meta = metadata[i]
        color = type_colors.get(meta['type'], 'gray')
        
        fig.add_trace(go.Scatter3d(
            x=[coords[0]],
            y=[coords[1]],
            z=[coords[2]],
            mode='markers+text',
            marker=dict(size=15, color=color, line=dict(color='black', width=2)),
            text=[meta['label']],
            textposition='top center',
            name=f"{meta['label']} ({meta['type']})",
            hovertext=f"{meta['label']}\nType: {meta['type']}\nFloor: {meta['floor']}",
            hoverinfo='text'
        ))
    
    fig.update_layout(
        title='Building Connectivity Graph (for Neo4j export)',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z (Floor)',
            aspectmode='data'
        ),
        width=900,
        height=600,
        legend=dict(x=1.02, y=1)
    )
    
    return fig

fig = visualize_graph_3d(graph, room_metadata)
fig.show()

## 6. Generate Cypher Queries

Here are the Cypher queries that would be used to import this data into Neo4j.

In [ ]:
def generate_cypher_create(nodes, relationships):
    """Generate Cypher CREATE statements."""
    statements = []
    
    # Create nodes
    statements.append("// Create Room nodes")
    for node in nodes:
        cypher = f"""CREATE (:{node['type'].title()} {{
    id: {node['id']},
    label: '{node['label']}',
    type: '{node['type']}',
    floor: {node['floor']},
    area: {node['area']},
    x: {node['x']},
    y: {node['y']},
    z: {node['z']}
}})"""
        statements.append(cypher)
    
    statements.append("\n// Create relationships")
    for rel in relationships:
        cypher = f"""MATCH (a {{label: '{rel['start_label']}'}}), (b {{label: '{rel['end_label']}'}})
CREATE (a)-[:{rel['type']}]->(b)"""
        statements.append(cypher)
    
    return "\n\n".join(statements)

cypher_create = generate_cypher_create(nodes, relationships)
print("Cypher CREATE Statements:")
print("=" * 60)
print(cypher_create)

In [ ]:
# Example Cypher query patterns for spatial analysis
print("Example Cypher Queries for Spatial Analysis:")
print("=" * 60)

queries = [
    ("Find all adjacent rooms", 
     "MATCH (a)-[:ADJACENT_TO]->(b) RETURN a.label, b.label"),
    
    ("Find rooms on floor 0",
     "MATCH (r) WHERE r.floor = 0 RETURN r.label, r.type"),
    
    ("Find vertically connected rooms",
     "MATCH (a)-[:STACKED_ON]->(b) RETURN a.label AS upper, b.label AS lower"),
    
    ("Shortest path between two rooms",
     "MATCH path = shortestPath((a {label: 'Room_A'})-[*]-(b {label: 'Room_F'}))\nRETURN path"),
    
    ("Count rooms by type",
     "MATCH (r) RETURN r.type, count(*) AS count ORDER BY count DESC"),
    
    ("Find all rooms within 2 hops of Room_A",
     "MATCH (start {label: 'Room_A'})-[*1..2]-(connected)\nRETURN DISTINCT connected.label"),
]

for title, query in queries:
    print(f"\n-- {title}")
    print(query)

## 7. Neo4j Driver Example (Optional)

If you have Neo4j installed and running, here's how you would connect and execute queries.

In [ ]:
# NOTE: This cell requires a running Neo4j instance
# Uncomment and modify the connection details to test

if NEO4J_AVAILABLE:
    print("Neo4j connection example (not executed - requires running Neo4j):")
    print()
    print('''
# Connect to Neo4j
url = "bolt://localhost:7687"
username = "neo4j"
password = "your_password"

driver = GraphDatabase.driver(url, auth=(username, password))

# Clear existing data
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

# Create nodes and relationships
with driver.session() as session:
    for node in nodes:
        session.run("""
            CREATE (r:Room {
                id: $id, label: $label, type: $type,
                floor: $floor, area: $area,
                x: $x, y: $y, z: $z
            })
        """, **node)
    
    for rel in relationships:
        session.run("""
            MATCH (a:Room {label: $start_label})
            MATCH (b:Room {label: $end_label})
            CREATE (a)-[:" + rel['type'] + "]->(b)
        """, start_label=rel['start_label'], end_label=rel['end_label'])

# Query example
with driver.session() as session:
    result = session.run("MATCH (a)-[r]->(b) RETURN a.label, type(r), b.label")
    for record in result:
        print(record)

driver.close()
''')
else:
    print("Neo4j driver not installed.")
    print("Install with: pip install neo4j")

## 8. Export Graph Data to JSON

Export the graph structure to JSON for later import into Neo4j or other systems.

In [ ]:
# Create exportable graph structure
graph_export = {
    "nodes": nodes,
    "relationships": relationships,
    "metadata": {
        "source": "topologic_fast",
        "num_nodes": len(nodes),
        "num_relationships": len(relationships),
        "graph_density": graph.Density(),
        "graph_diameter": graph.Diameter()
    }
}

# Save to file
with open("./building_graph.json", "w") as f:
    json.dump(graph_export, f, indent=2)

print("Exported graph to building_graph.json")
print(f"\nGraph Summary:")
print(json.dumps(graph_export["metadata"], indent=2))

## 9. Larger Example: Office Floor Plan

Let's create a more complex example similar to the floorplan_graph notebook.

In [ ]:
# Create a larger office layout
floor_height = 3.0

office_rooms = []
office_metadata = []

def add_room(x, y, w, l, name, rtype):
    room = tf.Cell.Box(x, y, 0, w, l, floor_height)
    office_rooms.append(room)
    office_metadata.append({
        "label": name,
        "type": rtype,
        "floor": 0,
        "area": w * l
    })

# Create rooms (simplified layout)
add_room(0, 0, 5, 4, "Reception", "reception")
add_room(5, 0, 10, 4, "Lobby", "lobby")
add_room(0, 4, 15, 4, "Open_Office", "workspace")
add_room(0, 8, 5, 4, "Conference_A", "conference")
add_room(5, 8, 5, 4, "Kitchen", "kitchen")
add_room(10, 8, 5, 4, "Conference_B", "conference")

# Create CellComplex
office_building = tf.CellComplex.ByCells(office_rooms)
office_graph = tf.Graph.ByTopology(office_building)

print(f"Office Layout:")
print(f"  Rooms: {office_building.NumCells()}")
print(f"  Connections: {office_graph.Size()}")
print(f"  Graph Diameter: {office_graph.Diameter()}")

In [ ]:
# Visualize the office graph
fig = visualize_graph_3d(office_graph, office_metadata)
fig.update_layout(title='Office Floor Plan Graph (for Neo4j export)')
fig.show()

## Summary

This notebook demonstrated:

1. **Creating Topology** - Building CellComplex structures with topologic_fast
2. **Graph Generation** - Creating dual graphs using `tf.Graph.ByTopology()`
3. **Data Extraction** - Extracting nodes and relationships for Neo4j
4. **Cypher Generation** - Creating Cypher queries for Neo4j import
5. **JSON Export** - Saving graph data for external use

### Features Not Yet Implemented in topologic_fast:

- `tf.Neo4j` - Native Neo4j integration class
- `tf.Dictionary` - Attaching metadata to topologies
- `tf.Topology.SetDictionary()` / `tf.Topology.Dictionary()` - Dictionary operations

### Next Steps:

To use this with Neo4j:
1. Install Neo4j Desktop or use Neo4j Aura (cloud)
2. Install the Python driver: `pip install neo4j`
3. Use the generated Cypher queries to import the data
4. Use Neo4j Browser or Bloom for visualization and querying